# Comparativo de Resultados - Benchmarks do Logscan

Este notebook lê os arquivos gerados pelo Logscan (que começam com `benchmark_`) e apresenta gráficos comparativos de Parsing Accuracy (PA), FTA, GA e FGA separados por dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

csv_files = glob.glob('benchmark_*.csv')

metrics = ['Accuracy', 'FTA', 'GA', 'FGA', 'Time', 'LLM Calls']
import re
import numpy as np

def parse_time_to_seconds(t_str):
    if pd.isna(t_str) or not isinstance(t_str, str): return np.nan
    match = re.search(r'(\d+)h\s+(\d+)m\s+([0-9.]+)s', t_str)
    if match:
        h = int(match.group(1))
        m = int(match.group(2))
        s = float(match.group(3))
        return h * 3600 + m * 60 + s
    return np.nan

data_dict_no_test = {m: {} for m in metrics}
data_dict_test = {m: {} for m in metrics}

for file in csv_files:
    filename = os.path.basename(file)
    label = filename.replace('benchmark_', '').replace('.csv', '')
    try:
        df = pd.read_csv(file, index_col='Dataset')
        for metric in metrics:
            if metric in df.columns:
                series = df[metric]
                if metric == 'Time':
                    series = series.apply(parse_time_to_seconds)
                if 'test' in label.lower():
                    data_dict_test[metric][label] = series
                else:
                    data_dict_no_test[metric][label] = series
    except Exception as e:
        print(f"Ignorando {file}: {e}")


In [ ]:
def plot_metrics(data_dict, title_suffix):
    for metric in metrics:
        if not data_dict[metric]:
            continue
        df_plot = pd.DataFrame(data_dict[metric])
        if df_plot.empty:
            continue
        cols = list(df_plot.columns)
        if 'LILAC' in cols:
            cols.remove('LILAC')
            cols.append('LILAC')
            df_plot = df_plot[cols]

        ax = df_plot.plot(kind='bar', figsize=(16, 6), width=0.8, colormap='tab10')
        metric_name = 'Parsing Accuracy' if metric == 'Accuracy' else metric

        plt.title(f'Comparativo de {metric_name} {title_suffix}', fontsize=16, fontweight='bold')
        plt.ylabel(metric_name, fontsize=14)
        plt.xlabel('Dataset', fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=12)
        if metric not in ['Time', 'LLM Calls']:
            plt.ylim([0, 1.05])
        plt.legend(title='Versão / Configuração', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

plot_metrics(data_dict_no_test, "")


In [ ]:
plot_metrics(data_dict_test, "(Test run)")


In [ ]:
# Tabela Consolidada (Logscan vs LILAC)
import numpy as np

# Ordenação oficial dos datasets da literatura
datasets_order = ['Hadoop', 'HDFS', 'OpenStack', 'Spark', 'Zookeeper', 'BGL', 
                  'HPC', 'Thunderbird', 'Linux', 'Mac', 'Apache', 'OpenSSH', 
                  'HealthApp', 'Proxifier']

metrics_order = ['GA', 'FGA', 'Accuracy', 'FTA', 'Time', 'LLM Calls']
metric_names = ['GA', 'FGA', 'PA', 'FTA', 'Time (s)', 'LLM Calls']

# Filtrando quais versões do Logscan nós processamos
logscan_versions = [k for k in data_dict_no_test['Accuracy'].keys() if k != 'LILAC']
parsers = logscan_versions + ['LILAC']

# Construindo as colunas MultiIndex para agrupar por Parser > Métrica
col_tuples = []
for parser in parsers:
    for m in metric_names:
        col_tuples.append((parser, m))

df_table = pd.DataFrame(index=datasets_order, columns=pd.MultiIndex.from_tuples(col_tuples))

# Populando os dados
for parser in parsers:
    for orig_m, new_m in zip(metrics_order, metric_names):
        try:
            series = data_dict_no_test[orig_m][parser]
            if orig_m in ['Time', 'LLM Calls']:
                df_table[(parser, new_m)] = series
            else:
                # Multiplicando por 100 para bater com a % da tabela oficial
                df_table[(parser, new_m)] = series * 100
        except KeyError:
            df_table[(parser, new_m)] = np.nan

# Limpando datasets vazios e reordenando seguindo o artigo
df_table = df_table.reindex(datasets_order)

# Aplicando a formatação e o Highlight dinâmico no maior valor da linha por métrica
styler = df_table.style.format('{:.1f}', na_rep='-')
for m in metric_names:
    styler = styler.highlight_max(
        subset=pd.IndexSlice[:, pd.IndexSlice[:, m]], 
        axis=1, 
        props='font-weight: bold;'
    )
styler.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    
display(styler)
